In [20]:
import numpy as np

In [69]:
def softmax(x):
    rows, cols = x.shape
    ans = []
    for i in range(rows):
        row = []
        for j in range(cols):
            row.append(0.0)
        ans.append(row)
        
    ans = np.array(ans)
    for i in range(rows):
        max_val = np.max(x[i])   
        row = x[i] - max_val
        
        exp = np.exp(row)
        sm = np.sum(exp)
        ans[i] = exp / sm
    
    return ans

In [71]:
def self_att(Q,K,V):
    d = Q.shape[-1]
    
    qkt = np.matmul(Q, K.T) / np.sqrt(d)
    softmaxqkt = softmax(qkt)
    att = np.matmul(softmaxqkt, V)
    return att

In [75]:
import numpy as np

def flash_att(Q,K,V,b_size=2):
    N, d = Q.shape
    
    # initialize O,m,l
    O = []
    for i in range(N):
        row = []
        for j in range(d):
            row.append(0.0)
        O.append(row)
    O = np.array(O)
    m = []
    for i in range(N):
        m.append(-np.inf)
    m = np.array(m)
    rowsm = []
    for i in range(N):
        rowsm.append(0.0)
    rowsm = np.array(rowsm)
    
    for j in range(0, N, b_size):
        Kb = K[j:j+b_size]
        Vb = V[j:j+b_size]
        
        for i in range(0, N, b_size):
            Qb = Q[i:i+b_size]
            m_old = m[i:i+b_size]
            rowsm_old = rowsm[i:i+b_size]
            O_old = O[i:i+b_size]
            temp = np.matmul(Qb, Kb.T)
            #print(temp)
            scores = temp / np.sqrt(d)
            max_row = np.max(scores, axis=1)
            m_new = np.maximum(m_old, max_row)
            #print(max_row,m_new)
            correction_factor = np.exp(m_old - m_new)

            exp_new = np.zeros_like(scores)

            for r in range(scores.shape[0]):      
                for c in range(scores.shape[1]):   
                    exp_new[r][c] = np.exp(scores[r][c] - m_new[r])

            rowsm_new = correction_factor * rowsm_old + np.sum(exp_new, axis=1)
            #print(rowsm_new)
            O_new = np.zeros_like(O_old)
            for r in range(O_old.shape[0]):
                for c in range(O_old.shape[1]):
                    O_new[r][c] = correction_factor[r] * O_old[r][c]

            O_new += np.matmul(exp_new, Vb)
            #print(O_new)           
      
            m[i:i+b_size] = m_new
            rowsm[i:i+b_size] = rowsm_new
            O[i:i+b_size] = O_new
    for i in range(N):
        O[i] = O[i] / rowsm[i]
    return O

In [72]:
def test(N,d,B):
    np.random.seed(0)
    Q = np.random.randn(N, d)
    K = np.random.randn(N, d)
    V = np.random.randn(N, d)
    out1 = self_att(Q, K, V)
    out2 = flash_att(Q, K, V, b_size=B)
    
    
    print(" Self-Attention")
    print(out1)
    print("Flash Attention ")
    print(out2)
    

In [76]:
test(4,4,2)
test(6,4,2)
test(8,4,2)
test(8,4,4)

 Self-Attention
[[-0.70811753 -0.9003788  -1.29057301  1.06773569]
 [-0.95848087 -1.44845917 -1.36636925  1.45121241]
 [-0.25175844 -0.4358699  -1.00409108  0.63642757]
 [-0.67161157 -1.05378635 -1.11568195  0.94056785]]
Flash Attention 
[[-0.70811753 -0.9003788  -1.29057301  1.06773569]
 [-0.95848087 -1.44845917 -1.36636925  1.45121241]
 [-0.25175844 -0.4358699  -1.00409108  0.63642757]
 [-0.67161157 -1.05378635 -1.11568195  0.94056785]]
 Self-Attention
[[-0.78107647 -0.69394194 -0.43693899  0.12079245]
 [-1.34354228 -0.28881136 -0.77886447  0.21760735]
 [-0.38651998 -0.39282209 -0.6489997   0.06829232]
 [-0.78661515 -0.46447673 -0.52366915 -0.09124141]
 [-1.11454636 -0.38966529 -0.66553623 -0.03783732]
 [-0.26601699 -0.04774501 -0.47498844 -0.16737374]]
Flash Attention 
[[-0.78107647 -0.69394194 -0.43693899  0.12079245]
 [-1.34354228 -0.28881136 -0.77886447  0.21760735]
 [-0.38651998 -0.39282209 -0.6489997   0.06829232]
 [-0.78661515 -0.46447673 -0.52366915 -0.09124141]
 [-1.11454636